In [0]:
%sql
select * from olist_dataset.silver.sellers

In [0]:
%sql
create or replace temporary view s1 as 

SELECT 
    seller_id,
    sum(total_price) as total_revenue,
    count(distinct  item_id) as total_items_sold,
    count(order_id) as total_orders,
    round(avg(overall_review),2) as rating_of_seller,
    max(est_shipping_date) as recent_shipping_date,
    min(est_shipping_date) as first_shipping_date
FROM olist_dataset.gold.items
GROUP BY seller_id

In [0]:
%sql
create or replace temporary view s2 as 

select 
    i.seller_id,
    cast(max(customer_delivery_date)as date) as recent_delivery_date,
    cast(min(customer_delivery_date)as date) as first_delivery_date
from olist_dataset.gold.items  as i
left join olist_dataset.silver.delivered_orders as o 
on o.order_id = i.order_id
group by seller_id

In [0]:
%sql
create or replace temporary view s3 as
select 
    seller_id,
    count(i.order_id) as faild_orders
from olist_dataset.gold.items as i 
left join olist_dataset.silver.undelivered_orders as o
on i.order_id = o.order_id
group by seller_id

In [0]:
%sql
create or replace temporary view final_s as 

select
    s.seller_id,
    cast(s.seller_zip_code_prefix as string) as seller_zip_code_prefix,
    s.seller_city,
    s.seller_state,
    s1.total_revenue,
    s1.total_items_sold,
    s1.total_orders,
    s1.rating_of_seller,
    s1.recent_shipping_date,
    s1.first_shipping_date,
    s2.recent_delivery_date,
    s2.first_delivery_date,
    s3.faild_orders
from olist_dataset.silver.sellers as s
left join s1 on s.seller_id = s1.seller_id
left join s2 on s.seller_id = s2.seller_id
left join s3 on s.seller_id = s3.seller_id

In [0]:
df = spark.read.table('final_s')

df = df.fillna({'faild_orders' : 0})

In [0]:
df.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema', True)\
    .saveAsTable('olist_dataset.gold.sellers')